### Plaques marking alignment

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import scanpy as sc
import tifffile as tiff
import matplotlib.pyplot as plt

from scipy.ndimage import distance_transform_edt
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label, regionprops_table, find_contours
from skimage.morphology import (
    binary_closing,
    binary_opening,
    disk,
    remove_small_holes,
    remove_small_objects,
)


In [2]:
BASE_RESULTS_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627"
)

ALIGNMENT_DIR = BASE_RESULTS_DIR / "06_plaques_image_alignment"
OUTPUT_DIR = BASE_RESULTS_DIR / "07_plaque_15um_polygons"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    "APP_infected": {
        "h5ad_path": ALIGNMENT_DIR / "APP_infected_morphology_alignment" / "APP_infected_with_morphology_alignment.h5ad",
    },
    "APP_mock": {
        "h5ad_path": ALIGNMENT_DIR / "APP_mock_morphology_alignment" / "APP_mock_with_morphology_alignment.h5ad",
    },
}

DISPLAY_LEVEL = 2

# Set this to your best Z plane if needed.
# If None, it uses the z_index saved in the aligned h5ad.
OVERRIDE_Z_INDEX = None

PLAQUE_BUFFER_UM = 15.0

# Tune these after viewing QC overlays.
PLAQUE_SMOOTH_SIGMA = 1.0
THRESHOLD_METHOD = "quantile"  # "quantile" or "otsu"
PLAQUE_INTENSITY_QUANTILE = 0.997
PLAQUE_OTSU_MULTIPLIER = 1.25
MIN_PLAQUE_AREA_UM2 = 20.0
MAX_HOLE_AREA_UM2 = 20.0
PLAQUE_OPENING_RADIUS_PX = 1
PLAQUE_CLOSING_RADIUS_PX = 2

for sample_name, cfg in SAMPLES.items():
    print(sample_name, cfg["h5ad_path"], cfg["h5ad_path"].exists())

print("Output:", OUTPUT_DIR)

APP_infected /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/06_plaques_image_alignment/APP_infected_morphology_alignment/APP_infected_with_morphology_alignment.h5ad True
APP_mock /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/06_plaques_image_alignment/APP_mock_morphology_alignment/APP_mock_with_morphology_alignment.h5ad True
Output: /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/07_plaque_15um_polygons


### Helpers

In [3]:
def robust_rescale(img, lower=1, upper=99.8):
    img = img.astype(float, copy=False)
    lo, hi = np.nanpercentile(img, [lower, upper])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(img, dtype=float)
    return np.clip((img - lo) / (hi - lo), 0, 1)


def _maybe_float(value):
    try:
        return float(value) if value is not None else None
    except ValueError:
        return None


def _maybe_int(value):
    try:
        return int(value) if value is not None else None
    except ValueError:
        return None


def parse_ome_pixels(ome_xml):
    if not ome_xml:
        return {}

    root = ET.fromstring(ome_xml)
    if root.tag.startswith("{"):
        ns = root.tag.split("}")[0].strip("{")
        pixels = root.find(".//ome:Pixels", {"ome": ns})
    else:
        pixels = root.find(".//Pixels")

    if pixels is None:
        return {}

    attrs = pixels.attrib
    return {
        "size_x": _maybe_int(attrs.get("SizeX")),
        "size_y": _maybe_int(attrs.get("SizeY")),
        "size_z": _maybe_int(attrs.get("SizeZ")),
        "physical_size_x": _maybe_float(attrs.get("PhysicalSizeX")),
        "physical_size_y": _maybe_float(attrs.get("PhysicalSizeY")),
        "physical_size_x_unit": attrs.get("PhysicalSizeXUnit"),
        "physical_size_y_unit": attrs.get("PhysicalSizeYUnit"),
    }


def get_yx_shape(shape, axes):
    axes = (axes or "").upper()
    if len(axes) == len(shape) and "Y" in axes and "X" in axes:
        return int(shape[axes.index("Y")]), int(shape[axes.index("X")])
    return int(shape[-2]), int(shape[-1])


def inspect_ome_tiff(path):
    with tiff.TiffFile(path) as tif:
        series = tif.series[0]
        levels = getattr(series, "levels", None) or [series]
        ome_pixels = parse_ome_pixels(tif.ome_metadata)

        level_records = []
        for i, level in enumerate(levels):
            axes = getattr(level, "axes", getattr(series, "axes", ""))
            shape = tuple(int(v) for v in level.shape)
            y, x = get_yx_shape(shape, axes)
            level_records.append(
                {
                    "level": i,
                    "axes": axes,
                    "shape": shape,
                    "height_px": y,
                    "width_px": x,
                }
            )

    full_y = int(ome_pixels.get("size_y") or level_records[0]["height_px"])
    full_x = int(ome_pixels.get("size_x") or level_records[0]["width_px"])

    for record in level_records:
        record["downsample_y"] = full_y / record["height_px"]
        record["downsample_x"] = full_x / record["width_px"]

    return {
        **ome_pixels,
        "fullres_shape_yx": (full_y, full_x),
        "levels": level_records,
    }


def read_ome_level_single_z(path, level_index=2, z_index=0):
    with tiff.TiffFile(path) as tif:
        series = tif.series[0]
        levels = getattr(series, "levels", None) or [series]
        level = levels[min(level_index, len(levels) - 1)]
        arr = np.asarray(level.asarray())
        axes = level.axes.upper()

    if "Z" in axes:
        z_axis = axes.index("Z")
        z_index = min(int(z_index), arr.shape[z_axis] - 1)
        arr = np.take(arr, z_index, axis=z_axis)
        axes = axes.replace("Z", "")

    if len(axes) == arr.ndim and "Y" in axes and "X" in axes:
        y_axis = axes.index("Y")
        x_axis = axes.index("X")
        other_axes = [i for i in range(arr.ndim) if i not in (y_axis, x_axis)]
        arr = np.transpose(arr, other_axes + [y_axis, x_axis])

    img = np.squeeze(arr)
    if img.ndim > 2:
        img = np.nanmax(img, axis=tuple(range(img.ndim - 2)))

    if img.ndim != 2:
        raise ValueError(f"Expected 2D image after selecting Z={z_index}, got {img.shape}")

    return img


def apply_affine_xy(xy, matrix):
    xy = np.asarray(xy, dtype=float)
    xy_h = np.c_[xy, np.ones(xy.shape[0])]
    out = xy_h @ np.asarray(matrix, dtype=float).T
    return out[:, :2]


def get_alignment_info(adata):
    info = dict(adata.uns["morphology_alignment"])
    info["morphology_path"] = Path(str(info["morphology_path"]))
    info["image_to_xenium_matrix"] = np.asarray(info["image_to_xenium_matrix"], dtype=float)
    info["xenium_to_image_matrix"] = np.asarray(info["xenium_to_image_matrix"], dtype=float)
    return info


def get_xenium_xy(adata):
    if {"x_centroid", "y_centroid"}.issubset(adata.obs.columns):
        return adata.obs[["x_centroid", "y_centroid"]].to_numpy(float)
    if "X_spatial" in adata.obsm:
        return np.asarray(adata.obsm["X_spatial"][:, :2], dtype=float)
    if "spatial" in adata.obsm:
        return np.asarray(adata.obsm["spatial"][:, :2], dtype=float)
    raise KeyError("No spatial coordinate columns found.")


def get_cell_fullres_px(adata, alignment_info):
    if "morphology_fullres_px" in adata.obsm:
        return np.asarray(adata.obsm["morphology_fullres_px"][:, :2], dtype=float)

    xenium_xy = get_xenium_xy(adata)
    return apply_affine_xy(xenium_xy, alignment_info["xenium_to_image_matrix"])


def fullres_to_level_points(xy_fullres_px, level_info):
    return np.asarray(xy_fullres_px, dtype=float) / np.array(
        [level_info["downsample_x"], level_info["downsample_y"]],
        dtype=float,
    )

### Plaque segmentation and 15um assignment


In [4]:
def segment_plaques_from_image(img, ome_meta, level_info):
    pixel_area_um2 = (
        float(ome_meta["physical_size_x"])
        *float(ome_meta["physical_size_y"])
        *float(level_info["downsample_x"])
        *float(level_info["downsample_y"])
    ) 
    min_area_px = max(1, int(np.ceil(MIN_PLAQUE_AREA_UM2 / pixel_area_um2)))
    hole_area_px = max(1, int(np.ceil(MAX_HOLE_AREA_UM2 / pixel_area_um2)))

    img_scaled = robust_rescale(img)
    smoothed = gaussian(img_scaled, sigma=PLAQUE_SMOOTH_SIGMA, preserve_range=True)

    if THRESHOLD_METHOD == "quantile":
        threshold = float(np.nanquantile(smoothed, PLAQUE_INTENSITY_QUANTILE))
    elif THRESHOLD_METHOD == "otsu":
        threshold = float(threshold_otsu(smoothed) * PLAQUE_OTSU_MULTIPLIER)
    else:
        raise ValueError("THRESHOLD_METHOD must be 'quantile' or 'otsu'.")

    mask = smoothed > threshold

    if PLAQUE_OPENING_RADIUS_PX > 0:
        mask = binary_opening(mask, disk(PLAQUE_OPENING_RADIUS_PX))
    if PLAQUE_CLOSING_RADIUS_PX > 0:
        mask = binary_closing(mask, disk(PLAQUE_CLOSING_RADIUS_PX))

    mask = remove_small_objects(mask, min_size=min_area_px)
    mask = remove_small_holes(mask, area_threshold=hole_area_px)

    labels = label(mask)

    props = pd.DataFrame(
        regionprops_table(
            labels,
            intensity_image=img_scaled,
            properties=[
                "label",
                "area",
                "centroid",
                "bbox",
                "mean_intensity",
                "max_intensity",
            ],
        )
    )

    if props.empty:
        return props, mask, labels, threshold

    props = props.rename(
        columns={
            "label": "plaque_id",
            "area": "area_level_px",
            "centroid-0": "centroid_y_level_px",
            "centroid-1": "centroid_x_level_px",
            "bbox-0": "bbox_y0_level_px",
            "bbox-1": "bbox_x0_level_px",
            "bbox-2": "bbox_y1_level_px",
            "bbox-3": "bbox_x1_level_px",
        }
    )

    props["plaque_id"] = props["plaque_id"].astype(int)
    props["area_um2"] = props["area_level_px"] * pixel_area_um2
    props["threshold"] = threshold

    return props, mask, labels, threshold


def add_plaque_centroids(plaque_df, level_info, image_to_xenium_matrix):
    if plaque_df.empty:
        return plaque_df

    centroid_level_xy = plaque_df[
        ["centroid_x_level_px", "centroid_y_level_px"]
    ].to_numpy(float)

    centroid_fullres_xy = centroid_level_xy * np.array(
        [level_info["downsample_x"], level_info["downsample_y"]],
        dtype=float,
    )

    centroid_xenium_xy = apply_affine_xy(
        centroid_fullres_xy,
        image_to_xenium_matrix,
    )

    plaque_df = plaque_df.copy()
    plaque_df["centroid_x_fullres_px"] = centroid_fullres_xy[:, 0]
    plaque_df["centroid_y_fullres_px"] = centroid_fullres_xy[:, 1]
    plaque_df["centroid_x_xenium"] = centroid_xenium_xy[:, 0]
    plaque_df["centroid_y_xenium"] = centroid_xenium_xy[:, 1]

    return plaque_df


def assign_cells_to_plaque_buffer(adata, labels, plaque_df, cell_level_xy, ome_meta, level_info):
    adata.obs["nearest_plaque_id"] = pd.Series(pd.NA, index=adata.obs_names, dtype="Int64")
    adata.obs["nearest_plaque_distance_um"] = np.nan
    adata.obs["in_plaque_15um_polygon"] = False
    adata.obs["plaque_15um_polygon_id"] = pd.Series(pd.NA, index=adata.obs_names, dtype="Int64")

    if plaque_df.empty or labels.max() == 0:
        buffer_mask = np.zeros_like(labels, dtype=bool)
        distance_um = np.full(labels.shape, np.nan)
        return adata, buffer_mask, distance_um

    pixel_sampling_um = (
        float(ome_meta["physical_size_y"]) * float(level_info["downsample_y"]),
        float(ome_meta["physical_size_x"]) * float(level_info["downsample_x"]),
    )

    distance_um, nearest_indices = distance_transform_edt(
        labels == 0,
        sampling=pixel_sampling_um,
        return_indices=True,
    )

    nearest_label_image = labels[nearest_indices[0], nearest_indices[1]]
    buffer_mask = distance_um <= PLAQUE_BUFFER_UM

    xy = np.asarray(cell_level_xy, dtype=float)
    xi = np.rint(xy[:, 0]).astype(int)
    yi = np.rint(xy[:, 1]).astype(int)

    valid = (
        np.isfinite(xy).all(axis=1)
        & (xi >= 0)
        & (yi >= 0)
        & (xi < labels.shape[1])
        & (yi < labels.shape[0])
    )

    valid_idx = np.where(valid)[0]

    cell_distance = np.full(adata.n_obs, np.nan)
    cell_nearest_label = np.full(adata.n_obs, -1, dtype=int)
    cell_in_buffer = np.zeros(adata.n_obs, dtype=bool)

    cell_distance[valid_idx] = distance_um[yi[valid_idx], xi[valid_idx]]
    cell_nearest_label[valid_idx] = nearest_label_image[yi[valid_idx], xi[valid_idx]]
    cell_in_buffer[valid_idx] = buffer_mask[yi[valid_idx], xi[valid_idx]]

    nearest_series = pd.Series(pd.NA, index=adata.obs_names, dtype="Int64")
    nearest_series.iloc[valid_idx] = cell_nearest_label[valid_idx]
    nearest_series = nearest_series.mask(nearest_series <= 0, pd.NA).astype("Int64")

    proximal_series = pd.Series(pd.NA, index=adata.obs_names, dtype="Int64")
    proximal_idx = np.where(cell_in_buffer & (cell_nearest_label > 0))[0]
    proximal_series.iloc[proximal_idx] = cell_nearest_label[proximal_idx]

    adata.obs["nearest_plaque_id"] = nearest_series
    adata.obs["nearest_plaque_distance_um"] = cell_distance
    adata.obs["in_plaque_15um_polygon"] = cell_in_buffer
    adata.obs["plaque_15um_polygon_id"] = proximal_series.astype("Int64")

    counts = adata.obs.loc[
        adata.obs["in_plaque_15um_polygon"],
        "plaque_15um_polygon_id",
    ].value_counts()

    plaque_df["n_cells_in_15um_polygon"] = (
        plaque_df["plaque_id"].map(counts).fillna(0).astype(int)
    )

    return adata, buffer_mask, distance_um

### Save polygon contours and QC image

In [5]:
def save_buffer_contours(buffer_mask, level_info, output_path):
    contours = find_contours(buffer_mask.astype(float), 0.5)

    rows = []
    for polygon_id, contour in enumerate(contours):
        for vertex_id, (y_level, x_level) in enumerate(contour):
            rows.append(
                {
                    "polygon_id": polygon_id,
                    "vertex_id": vertex_id,
                    "x_level_px": x_level,
                    "y_level_px": y_level,
                    "x_fullres_px": x_level * level_info["downsample_x"],
                    "y_fullres_px": y_level * level_info["downsample_y"],
                }
            )

    pd.DataFrame(rows).to_csv(output_path, index=False)


def plot_plaque_qc(
    img,
    plaque_mask,
    buffer_mask,
    cell_level_xy,
    cell_in_buffer,
    plaque_df,
    output_path,
    title,
):
    fig, ax = plt.subplots(figsize=(12, 12))

    ax.imshow(robust_rescale(img), cmap="gray", origin="upper")

    ax.imshow(
        np.ma.masked_where(~buffer_mask, buffer_mask),
        cmap="autumn",
        alpha=0.25,
        origin="upper",
    )

    ax.imshow(
        np.ma.masked_where(~plaque_mask, plaque_mask),
        cmap="cool",
        alpha=0.45,
        origin="upper",
    )

    xy = np.asarray(cell_level_xy, dtype=float)
    valid = np.isfinite(xy).all(axis=1)

    background_cells = valid & ~cell_in_buffer
    proximal_cells = valid & cell_in_buffer

    rng = np.random.default_rng(0)
    background_idx = np.where(background_cells)[0]
    if len(background_idx) > 25000:
        background_idx = rng.choice(background_idx, 25000, replace=False)

    ax.scatter(
        xy[background_idx, 0],
        xy[background_idx, 1],
        s=0.15,
        c="white",
        alpha=0.20,
        linewidths=0,
        rasterized=True,
        label="other cells",
    )

    ax.scatter(
        xy[proximal_cells, 0],
        xy[proximal_cells, 1],
        s=0.55,
        c="lime",
        alpha=0.90,
        linewidths=0,
        rasterized=True,
        label="cells in plaque + 15 um polygon",
    )

    if not plaque_df.empty:
        ax.scatter(
            plaque_df["centroid_x_level_px"],
            plaque_df["centroid_y_level_px"],
            s=18,
            c="red",
            edgecolors="black",
            linewidths=0.3,
            label="plaque centroids",
        )

    ax.set_title(title)
    ax.set_xlim(0, img.shape[1])
    ax.set_ylim(img.shape[0], 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.legend(frameon=False, loc="lower right")

    fig.tight_layout()
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.show()

### Run on APP infected and APP mock

In [ ]:
summary_rows = []
adata_outputs = {}

for sample_name, cfg in SAMPLES.items():
    print(f"\n=== Processing {sample_name} ===")

    sample_out = OUTPUT_DIR / sample_name
    sample_out.mkdir(parents=True, exist_ok=True)

    adata_sample = sc.read_h5ad(cfg["h5ad_path"])
    alignment_info = get_alignment_info(adata_sample)

    morphology_path = alignment_info["morphology_path"]
    z_index = int(OVERRIDE_Z_INDEX) if OVERRIDE_Z_INDEX is not None else int(alignment_info.get("z_index", 0))
    display_level = int(DISPLAY_LEVEL)

    print("h5ad:", cfg["h5ad_path"])
    print("morphology:", morphology_path)
    print("z_index:", z_index)
    print("cells:", adata_sample.n_obs)

    ome_meta = inspect_ome_tiff(morphology_path)
    level_info = ome_meta["levels"][min(display_level, len(ome_meta["levels"]) - 1)]

    display_img = read_ome_level_single_z(
        morphology_path,
        level_index=display_level,
        z_index=z_index,
    )

    cell_fullres_px = get_cell_fullres_px(adata_sample, alignment_info)
    cell_level_xy = fullres_to_level_points(cell_fullres_px, level_info)

    adata_sample.obsm["morphology_plaque_level_px"] = cell_level_xy

    plaque_df, plaque_mask, plaque_labels, threshold = segment_plaques_from_image(
        display_img,
        ome_meta,
        level_info,
    )

    plaque_df = add_plaque_centroids(
        plaque_df,
        level_info,
        alignment_info["image_to_xenium_matrix"],
    )

    plaque_df.insert(0, "sample", sample_name)
    plaque_df["z_index"] = z_index
    plaque_df["display_level"] = display_level
    plaque_df["buffer_um"] = PLAQUE_BUFFER_UM

    adata_sample, buffer_mask, distance_um = assign_cells_to_plaque_buffer(
        adata_sample,
        plaque_labels,
        plaque_df,
        cell_level_xy,
        ome_meta,
        level_info,
    )

    adata_sample.uns["plaque_15um_analysis"] = {
        "sample_name": sample_name,
        "morphology_path": str(morphology_path),
        "z_index": int(z_index),
        "display_level": int(display_level),
        "plaque_buffer_um": float(PLAQUE_BUFFER_UM),
        "threshold_method": str(THRESHOLD_METHOD),
        "threshold": float(threshold),
        "plaque_intensity_quantile": float(PLAQUE_INTENSITY_QUANTILE),
        "plaque_otsu_multiplier": float(PLAQUE_OTSU_MULTIPLIER),
        "min_plaque_area_um2": float(MIN_PLAQUE_AREA_UM2),
        "max_hole_area_um2": float(MAX_HOLE_AREA_UM2),
        "n_plaques": int(len(plaque_df)),
        "n_cells_in_plaque_15um_polygon": int(adata_sample.obs["in_plaque_15um_polygon"].sum()),
    }

    plaque_csv = sample_out / f"{sample_name}_plaques_segmented.csv"
    h5ad_out = sample_out / f"{sample_name}_with_plaque_15um_cells.h5ad"
    qc_png = sample_out / f"{sample_name}_qc_plaque_15um_overlay.png"
    mask_npy = sample_out / f"{sample_name}_plaque_mask_level{display_level}.npy"
    buffer_npy = sample_out / f"{sample_name}_plaque_15um_buffer_mask_level{display_level}.npy"
    contours_csv = sample_out / f"{sample_name}_plaque_15um_buffer_contours.csv"

    plaque_df.to_csv(plaque_csv, index=False)
    np.save(mask_npy, plaque_mask)
    np.save(buffer_npy, buffer_mask)
    save_buffer_contours(buffer_mask, level_info, contours_csv)

    plot_plaque_qc(
        display_img,
        plaque_mask,
        buffer_mask,
        cell_level_xy,
        adata_sample.obs["in_plaque_15um_polygon"].to_numpy(bool),
        plaque_df,
        qc_png,
        title=f"{sample_name}: plaques and cells in 15 um plaque polygons",
    )

    adata_sample.write_h5ad(h5ad_out)
    adata_outputs[sample_name] = adata_sample

    row = {
        "sample": sample_name,
        "n_cells": int(adata_sample.n_obs),
        "n_plaques": int(len(plaque_df)),
        "n_cells_in_plaque_15um_polygon": int(adata_sample.obs["in_plaque_15um_polygon"].sum()),
        "percent_cells_in_plaque_15um_polygon": float(
            100 * adata_sample.obs["in_plaque_15um_polygon"].mean()
        ),
        "h5ad_output": str(h5ad_out),
        "plaque_csv": str(plaque_csv),
        "qc_image": str(qc_png),
        "contours_csv": str(contours_csv),
    }

    summary_rows.append(row)
    print(pd.Series(row).to_string())

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "plaque_15um_summary.csv", index=False)
display(summary_df)


=== Processing APP_infected ===
h5ad: /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/06_plaques_image_alignment/APP_infected_morphology_alignment/APP_infected_with_morphology_alignment.h5ad
morphology: /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/output-XETG00224__0023902__APPPS1_infected_453f30__20240627__192344/morphology.ome.tif
z_index: 3
cells: 55417


/var/folders/82/6l796mmn2fj1pv97hrt8rh1h0000gp/T/ipykernel_15405/947425486.py:24: FutureWarning: `binary_opening` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.opening` instead.
  mask = binary_opening(mask, disk(PLAQUE_OPENING_RADIUS_PX))
/var/folders/82/6l796mmn2fj1pv97hrt8rh1h0000gp/T/ipykernel_15405/947425486.py:26: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  mask = binary_closing(mask, disk(PLAQUE_CLOSING_RADIUS_PX))
/var/folders/82/6l796mmn2fj1pv97hrt8rh1h0000gp/T/ipykernel_15405/947425486.py:28: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the p

In [ ]:
print("Saved outputs under:")
print(OUTPUT_DIR)

for sample_name in SAMPLES:
    print(f"\n{sample_name}")
    sample_out = OUTPUT_DIR / sample_name
    for path in sorted(sample_out.glob("*")):
        print(" ", path.name)

In [ ]:
adata_sample.obs["in_plaque_15um_polygon"]
adata_sample.obs["plaque_15um_polygon_id"]
adata_sample.obs["nearest_plaque_id"]
adata_sample.obs["nearest_plaque_distance_um"]